# 도구 · MCP · A2A (Tools, MCP, A2A) 개념 이해
## LangChain 도구 · MCP · A2A 3패턴 · Individual Tool Agent

> 📦 **환경 설치·실행 명령**은 [`env_guides/M02_3_mcp_a2a.md`](env_guides/M02_3_mcp_a2a.md) 에 정리되어 있습니다.
> 반복·공통 코드는 [`agentic_lib/`](agentic_lib) 라이브러리(bootstrap·tools·mcp·a2a)로 분리해 두었습니다.

---

### 전제조건
- Windows 11 + **CMD(`cmd.exe`)** + Python **3.11**(`uv` 관리), 커널 = **`Agentic AI (uv)`**
- 로컬 LLM 서버 연결은 ([`M02_1_local_llm.ipynb`](M02_1_local_llm.ipynb))에서 먼저 확인
- 기본 LLM = **로컬 Ollama + `qwen3:8b`**(네이티브 도구 호출), 웹 검색 도구는 네트워크(DuckDuckGo) 필요

### 학습 목표
1. LangChain 기반 도구(Tool) 정의 및 에이전트 연결
2. MCP(Model Context Protocol) 아키텍처 이해 및 구현
3. A2A(Agent-to-Agent) 3가지 패턴 구현 (계층형 / 순차형 / 수평형)
4. 개인 도구 에이전트(Individual Tool Agent) 최종 완성

### 아키텍처 개요
```
[사용자]
   ↓
[MCP Host: LangChain Agent]
   ├─[MCP Client] ──→ [MCP Server: 검색 도구]
   ├─[MCP Client] ──→ [MCP Server: 계산기 도구]
   └─[로컬 LLM 서버: Ollama/llama.cpp · OpenAI 호환 /v1]
```


In [1]:
# [setup] 자기완결적 셋업 — 이 노트북만 단독으로 실행 가능하게 함.
import sys, os
sys.path.insert(0, os.path.abspath(''))   # notebooks/ 를 import 경로에 추가

import utils
utils.reload_env()   # .env 재로드 (LLM_PROVIDER 등 갱신) + 현재 공급자 상태 출력

from utils import get_llm, print_provider_status

# 이 파트는 도구·MCP·A2A 실습 → bootstrap·tools 에 더해 mcp·a2a 까지 사용
from agentic_lib import bootstrap, tools, mcp, a2a
from agentic_lib.bootstrap import to_text   # 공급자 무관 응답 정규화(<think> 제거 포함)

# 도구 실습에 필요한 패키지(웹 검색 ddgs 포함). 로컬/클라우드 모두 OpenAI 호환
utils.uv_install(['langchain', 'langchain-openai', 'langchain-community',
                  'langchain-google-genai', 'langchain-anthropic',
                  'openai', 'requests', 'ddgs'])


llm = utils.get_llm()   # LangChain BaseChatModel 반환 (기본 ollama/qwen3:8b)
print_provider_status()

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langchain', 'langchain-openai', 'langchain-community', 'langchain-google-genai', 'langchain-anthropic', 'openai', 'requests', 'ddgs']


LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


---
## 4. LangChain 도구(Tool) 정의

에이전트가 사용할 수 있는 도구를 정의합니다.

In [2]:
# §4 도구 — 공통 도구(agentic_lib.tools) 재사용 + 실제 웹 검색(DuckDuckGo)
from langchain_community.tools import DuckDuckGoSearchRun

# 실제 웹 검색 도구(네트워크 필요). 오프라인이면 tools.search_web(시뮬레이션)로 대체 가능.
search = DuckDuckGoSearchRun()

# agentic_lib.tools 의 공통 @tool 들을 그대로 사용 (calculator / get_current_time / get_weather)
agent_tools = [tools.calculator, tools.get_current_time, tools.get_weather, search]

print("등록된 도구:")
for t in agent_tools:
    print(f"  - {t.name}: {t.description[:60]}...")

등록된 도구:
  - calculator: 수학 수식을 계산합니다. 예: '2 ** 10', 'sqrt(144)', '(3+4)*5'.

    Pyt...
  - get_current_time: 현재 날짜와 시간을 'YYYY-MM-DD HH:MM:SS' 형식으로 반환합니다.

    인자가 없는 도구를...
  - get_weather: 도시의 현재 날씨를 조회합니다(무료 Open-Meteo, API 키 불필요). 예: '서울', '부산'.

...
  - duckduckgo_search: A wrapper around DuckDuckGo Search. Useful for when you need...


C:\Users\stshin\AppData\Local\Temp\ipykernel_40712\4088401716.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


---
## 5. MCP (Model Context Protocol) 아키텍처

### MCP 핵심 구성 요소
- **MCP Host:** AI 애플리케이션 (LangChain Agent)
- **MCP Client:** 서버와의 연결 유지 컴포넌트
- **MCP Server:** 컨텍스트(도구/데이터)를 제공하는 프로그램

```
[MCP Host]
  └─[MCP Client 1] ↔ [MCP Server: 파일 시스템]
  └─[MCP Client 2] ↔ [MCP Server: 웹 검색]
  └─[MCP Client 3] ↔ [MCP Server: 데이터베이스]
```

In [3]:
# MCP 인프라(MCPServer/Client/Host, Calculator/FileSystem 서버)는 agentic_lib.mcp 로 분리했습니다.
from agentic_lib.mcp import MCPHost, CalculatorMCPServer, FileSystemMCPServer

print("=== MCP 시스템 구성 ===")
host = MCPHost()
host.connect(CalculatorMCPServer())
host.connect(FileSystemMCPServer())

print("\n=== 사용 가능한 도구 목록 ===")
for server, tools_list in host.list_all_tools().items():
    print(f"\n[{server}]")
    for t in tools_list:
        print(f"  - {t['name']}: {t['description']}")

print("\n=== MCP 도구 호출 테스트 ===")
print("  계산 결과:", host.call_tool("calculator-server", "calculate", expression="2 ** 10"))
print("  파일 쓰기:", host.call_tool("filesystem-server", "write_file", path="memo.txt", content="MCP 테스트 파일"))
print("  파일 읽기:", host.call_tool("filesystem-server", "read_file", path="memo.txt"))

=== MCP 시스템 구성 ===
[MCP Client] 'calculator-server' 서버에 연결됨
[MCP Client] 'filesystem-server' 서버에 연결됨

=== 사용 가능한 도구 목록 ===

[calculator-server]
  - calculate: 수학 계산 수행
  - factorial: 팩토리얼 계산

[filesystem-server]
  - read_file: 파일 읽기
  - write_file: 파일 쓰기
  - list_files: 파일 목록 조회

=== MCP 도구 호출 테스트 ===
  [MCP] calculator-server.calculate({'expression': '2 ** 10'})
  계산 결과: {'result': 1024, 'expression': '2 ** 10'}
  [MCP] filesystem-server.write_file({'path': 'memo.txt', 'content': 'MCP 테스트 파일'})
  파일 쓰기: {'success': True, 'path': 'memo.txt'}
  [MCP] filesystem-server.read_file({'path': 'memo.txt'})
  파일 읽기: MCP 테스트 파일


---
## 6. A2A (Agent-to-Agent) 세 가지 패턴

### 패턴 1: 계층형 (Hierarchical)
대장 에이전트가 하위 에이전트에게 업무를 분배

In [4]:
# A2A 인프라(BaseAgent/CoordinatorAgent/PipelineAgent/SharedCanvas/PeerAgent)는 agentic_lib.a2a 로 분리.
from agentic_lib.a2a import BaseAgent, CoordinatorAgent

# 하위 전문 에이전트들
search_agent = BaseAgent("SearchAgent", "search specialist")
calc_agent = BaseAgent("CalculatorAgent", "math specialist")
file_agent = BaseAgent("FileAgent", "file management specialist")

coordinator = CoordinatorAgent("Coordinator", [search_agent, calc_agent, file_agent])

print("=== 계층형(Hierarchical) A2A 패턴 ===")
print("대장 에이전트가 작업을 하위 에이전트에 분배\n")
task = "AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘"
print(f"입력 태스크: {task}\n")
results = coordinator.delegate(task)
print(f"\n최종 결과: {results}")

=== 계층형(Hierarchical) A2A 패턴 ===
대장 에이전트가 작업을 하위 에이전트에 분배

입력 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘

  [Coordinator] → [SearchAgent]: 검색 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘
  [Coordinator] → [CalculatorAgent]: 계산 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘
  [Coordinator] → [FileAgent]: 파일 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저장해줘

최종 결과: {'search': "[SearchAgent] '검색 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저' 처리 완료", 'calc': "[CalculatorAgent] '계산 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저' 처리 완료", 'file': "[FileAgent] '파일 태스크: AI 트렌드를 검색하고, 예산 1000+500을 계산하고, 결과를 파일에 저' 처리 완료"}


In [5]:
# 패턴 2: 순차형 (Sequential / Pipeline) — PipelineAgent 는 agentic_lib.a2a 에 있음
from agentic_lib.a2a import PipelineAgent

# 파이프라인 각 단계의 처리 함수 (데모용 — 실제로는 LLM/도구 호출)
def collect_data(topic):
    """데이터 수집 단계."""
    return {"topic": topic,
            "raw_data": f"{topic}에 관한 원시 데이터: [기사1, 기사2, 기사3]",
            "source_count": 3}

def analyze_data(data):
    """데이터 분석 단계."""
    data["analysis"] = f"{data['source_count']}개 소스에서 주요 트렌드 추출 완료"
    data["keywords"] = ["AI", "에이전트", "LLM"]
    return data

def generate_report(data):
    """보고서 생성 단계."""
    return (f"=== {data['topic']} 분석 보고서 ===\n"
            f"분석: {data['analysis']}\n"
            f"키워드: {', '.join(data['keywords'])}")

def review_report(report):
    """보고서 검토 단계."""
    return report + "\n[검토 완료] 품질 검증 통과"

# 파이프라인 구성 및 연결 (set_next 가 다음 에이전트를 반환 → 체이닝)
collector = PipelineAgent("Collector", "데이터 수집", collect_data)
analyzer = PipelineAgent("Analyzer", "데이터 분석", analyze_data)
generator = PipelineAgent("Generator", "보고서 생성", generate_report)
reviewer = PipelineAgent("Reviewer", "보고서 검토", review_report)
collector.set_next(analyzer).set_next(generator).set_next(reviewer)

print("=== 순차형(Sequential) A2A 패턴 ===")
print("컨베이어 벨트처럼 순서대로 처리\n")
final_result = collector.execute("Agentic AI 트렌드")
print(f"\n최종 보고서:\n{final_result}")

=== 순차형(Sequential) A2A 패턴 ===
컨베이어 벨트처럼 순서대로 처리

  [Collector(데이터 수집)] 처리 중: Agentic AI 트렌드
    → 출력: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 원시 데이터: [기사1, 기사2, 기
  [Analyzer(데이터 분석)] 처리 중: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 
    → 출력: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 원시 데이터: [기사1, 기사2, 기
  [Generator(보고서 생성)] 처리 중: {'topic': 'Agentic AI 트렌드', 'raw_data': 'Agentic AI 트렌드에 관한 
    → 출력: === Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: AI, 에이전트, LLM
  [Reviewer(보고서 검토)] 처리 중: === Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: 
    → 출력: === Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: AI, 에이전트, LLM
[검토 완료

최종 보고서:
=== Agentic AI 트렌드 분석 보고서 ===
분석: 3개 소스에서 주요 트렌드 추출 완료
키워드: AI, 에이전트, LLM
[검토 완료] 품질 검증 통과


In [6]:
# 패턴 3: 수평형 (Peer-to-Peer / Shared Canvas) — SharedCanvas/PeerAgent 는 agentic_lib.a2a 에 있음
from agentic_lib.a2a import SharedCanvas, PeerAgent

# 전문 분야별 의견 생성 (데모용 규칙 — 실제로는 LLM 호출 클로저를 주입). (expertise, topic) -> str
def opinion(expertise, topic):
    """전문 분야별 의견 문자열을 돌려준다(시뮬레이션)."""
    book = {
        "기술": f"{topic}의 기술 스택은 LangGraph + 로컬 LLM(OpenAI 호환)이 최적입니다.",
        "비즈니스": f"{topic}의 비즈니스 가치는 업무 자동화로 연간 30% 비용 절감입니다.",
        "보안": f"{topic} 구현 시 NeMo Guardrails 로 입출력 필터링이 필수입니다.",
        "UX": f"{topic}의 사용자 경험은 응답 속도 < 2초를 목표로 해야 합니다.",
    }
    return book.get(expertise, f"{topic}에 대한 {expertise} 관점 의견")

# 공유 캔버스 + 수평형 에이전트들 (opinion_fn 으로 의견 생성기를 주입)
canvas = SharedCanvas()
peers = [
    PeerAgent("Tech Expert", "기술", canvas, opinion_fn=opinion),
    PeerAgent("Biz Expert", "비즈니스", canvas, opinion_fn=opinion),
    PeerAgent("Security Expert", "보안", canvas, opinion_fn=opinion),
    PeerAgent("UX Expert", "UX", canvas, opinion_fn=opinion),
]

print("=== 수평형(Peer-to-Peer) A2A 패턴 ===")
print("동등한 에이전트들이 공유 캔버스에서 협업\n")
topic = "회사 내부 AI 에이전트 도입 방안"
print(f"토론 주제: {topic}\n")
for agent in peers:
    agent.contribute(topic)
print(f"\n{canvas.get_summary()}")

=== 수평형(Peer-to-Peer) A2A 패턴 ===
동등한 에이전트들이 공유 캔버스에서 협업

토론 주제: 회사 내부 AI 에이전트 도입 방안

  [Tech Expert] 캔버스에 '기술' 의견 추가
  [Biz Expert] 캔버스에 '비즈니스' 의견 추가
  [Security Expert] 캔버스에 '보안' 의견 추가
  [UX Expert] 캔버스에 'UX' 의견 추가

[공유 캔버스 현황]
  [Tech Expert] 기술_의견: 회사 내부 AI 에이전트 도입 방안의 기술 스택은 LangGraph + 로컬 LLM(OpenAI 호환)이 최
  [Biz Expert] 비즈니스_의견: 회사 내부 AI 에이전트 도입 방안의 비즈니스 가치는 업무 자동화로 연간 30% 비용 절감입니다.
  [Security Expert] 보안_의견: 회사 내부 AI 에이전트 도입 방안 구현 시 NeMo Guardrails 로 입출력 필터링이 필수입니다.
  [UX Expert] UX_의견: 회사 내부 AI 에이전트 도입 방안의 사용자 경험은 응답 속도 < 2초를 목표로 해야 합니다.


---
## 7. LangChain 기반 실제 에이전트 구현

앞에서 정의한 도구들을 LangChain Agent 에 연결합니다.

> **도구 호출(tool calling) 공급자** — 기본값 **Ollama + `qwen3:8b`** 는 OpenAI 호환 `/v1` 에서 네이티브 `tool_calls` 를 안정적으로 생성하므로, 아래 셀은 `utils.get_llm()` 이 돌려주는 기본 공급자를 그대로 에이전트에 사용합니다. 로컬 `llamacpp`/`vllm` 처럼 function calling 이 불안정한 공급자를 쓸 때는 `.env` 의 `LLM_PROVIDER` 를 도구 호출이 안정적인 공급자(예: `ollama`, `google`)로 바꿔 주면 됩니다.
>
> 📓 **3자 반복 실험으로 직접 비교**: 로컬 소형(Qwen2.5-0.5B, 0%) vs **로컬 대형(Qwen3-8B, 100%)** vs 클라우드(100%) function calling 신뢰도를 N회 반복 실험으로 비교 → 보충 노트북 **`M02_2_function_calling.ipynb`**. **핵심**: 도구 호출 실패는 '로컬이라서'가 아니라 '모델이 작아서'이며, Qwen3-8B 같은 큰 모델이면 로컬에서도 안정적 function calling 이 가능합니다.

In [7]:
from langchain.agents import create_agent

system_prompt = """당신은 다양한 도구를 사용할 수 있는 유능한 AI 에이전트입니다.
사용자의 요청을 분석하고, 적절한 도구를 선택하여 정확한 답변을 제공하세요.

사용 가능한 도구:
- calculator: 수학 계산
- get_current_time: 현재 날짜/시간
- get_weather: 날씨 조회
- duckduckgo_search: 웹 검색

항상 한국어로 답변하세요."""

try:
    if llm is None:
        print("LLM 연결 필요: .env 의 공급자/키를 확인하세요.")
    else:
        # 기본 공급자(ollama/qwen3:8b)는 네이티브 도구 호출이 안정적 → get_llm 그대로 사용
        agent_llm = utils.get_llm()
        agent = create_agent(agent_llm, agent_tools, system_prompt=system_prompt)

        print("=== LangChain 에이전트 실행 ===")
        result = agent.invoke({
            "messages": [("human", "오늘 날짜를 알려주고, 서울의 날씨도 확인해줘. 그리고 100의 제곱근을 계산해줘.")]
        })
        final = to_text(result["messages"][-1].content)
        print(f"\n최종 답변: {final}")

except Exception as e:
    print(f"에이전트 실행 오류: {type(e).__name__}: {e}")
    print("LLM 백엔드/도구 호출 설정을 확인하세요.")

=== LangChain 에이전트 실행 ===


에이전트 실행 오류: Exception: [500] {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}
{'error': {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}, 'status': 500}
LLM 백엔드/도구 호출 설정을 확인하세요.


---
## 8. 최종 실습: Individual Tool Agent 구현

### 과제: 나만의 전문 에이전트 만들기
아래 템플릿을 활용해 특정 분야에 특화된 에이전트를 구현하세요.

In [8]:
from langchain.tools import tool
from langchain.agents import create_agent
import math
from datetime import datetime

# ============================================================
# TODO: 여기에 나만의 도구를 추가하세요!
# ============================================================

@tool
def my_custom_tool(input: str) -> str:
    """여기에 도구 설명을 작성하세요.
    어떤 입력을 받고 어떤 출력을 반환하는지 명확히 설명합니다.
    """
    return f"나의 도구 실행 결과: {input}"


@tool
def get_stock_price(symbol: str) -> str:
    """주식 종목의 현재 가격을 조회합니다.
    예: 'AAPL', 'GOOGL', 'NVDA'
    """
    mock_prices = {
        "AAPL": 182.5, "GOOGL": 175.3, "NVDA": 875.2,
        "MSFT": 415.7, "AMZN": 188.4,
    }
    price = mock_prices.get(symbol.upper(), 100.0)
    return f"{symbol.upper()} 현재 가격: ${price:.2f} USD (시뮬레이션)"


@tool
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """통화를 변환합니다. 지원 통화: USD, KRW, EUR, JPY"""
    rates = {
        "USD_KRW": 1340.0, "USD_EUR": 0.92, "USD_JPY": 149.5,
        "KRW_USD": 1/1340.0, "EUR_USD": 1/0.92,
    }
    key = f"{from_currency.upper()}_{to_currency.upper()}"
    rate = rates.get(key, 1.0)
    result = amount * rate
    return f"{amount} {from_currency} = {result:.2f} {to_currency} (환율: {rate})"


@tool
def calculator_fin(expression: str) -> str:
    """수학/금융 계산을 수행합니다. Python 수식을 입력하세요."""
    try:
        result = eval(expression, {"__builtins__": {}}, {"math": math})
        return f"{expression} = {result}"
    except Exception as e:
        return f"오류: {e}"


finance_tools = [get_stock_price, convert_currency, calculator_fin]

finance_system_prompt = """당신은 금융 전문 AI 에이전트입니다.
주식 가격 조회, 환율 계산, 수익률 계산 등을 도와드립니다.
항상 정확한 숫자와 함께 한국어로 답변하세요."""

try:
    if llm is None:
        print("LLM 연결 필요: API 키를 .env에 설정하세요.")
    else:
        # 기본 공급자(ollama/qwen3:8b)는 네이티브 도구 호출이 안정적 → get_llm 그대로 사용
        agent_llm = utils.get_llm()
        finance_agent = create_agent(agent_llm, finance_tools, system_prompt=finance_system_prompt)

        print("=== 금융 전문 에이전트 실행 ===")
        result = finance_agent.invoke({
            "messages": [("human", "NVIDIA 주식 가격을 알려주고, 100달러를 원화로 환산해줘. NVIDIA 10주의 총 가치는?")]
        })
        final = to_text(result["messages"][-1].content)
        print(f"\n최종 답변:\n{final}")

except Exception as e:
    print(f"에이전트 오류: {type(e).__name__}: {e}")

=== 금융 전문 에이전트 실행 ===


에이전트 오류: Exception: [500] {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}
{'error': {'message': 'Failed to generate completions: Failed to apply prompt template: invalid operation: This model only supports single tool-calls at once! (in tool_use:95)', 'type': 'Internal Server Error', 'code': 500}, 'status': 500}


---
## 9. 정리 및 다음 모듈 예고

### 핵심 정리
1. **로컬 LLM 서버:** Ollama(권장)·llama.cpp 모두 **OpenAI 호환 `/v1`** 을 제공 → 클라우드와 같은 코드로 사용, `LLM_PROVIDER` 한 줄로 전환
2. **MCP:** Host-Client-Server 구조로 에이전트와 외부 도구/데이터를 표준 방식으로 연결
3. **A2A 패턴:**
   - **계층형:** 대장 에이전트가 업무 분배 → 중앙 집중식
   - **순차형:** 파이프라인으로 단계별 처리 → 데이터 변환에 적합
   - **수평형:** 공유 캔버스에서 협업 → 창의적 작업에 적합
4. **LangChain Agent:** 도구 + LLM + 프롬프트를 결합한 실용적 에이전트 (qwen3:8b 네이티브 도구 호출)

---
### 참고 자료
- 환경 가이드: [`env_guides/M02_3_mcp_a2a.md`](env_guides/M02_3_mcp_a2a.md)
- 공통 라이브러리: `agentic_lib` (bootstrap·tools·mcp·a2a)
- MCP 공식 문서: https://modelcontextprotocol.io
- LangChain 에이전트 가이드: https://python.langchain.com/docs/how_to/#agents